<table align="left">
  <td>
    <a href="https://colab.research.google.com/" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab" title="Abrir y ejecutar en Google Colaboratory"/></a>
  </td>
  <td>
    <a target="_blank" href="https://kaggle.com/kernels/welcome"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" alt="Abrir en Kaggle" title="Abrir y ejecutar en Kaggle"/></a>
  </td>
</table>

<br><br>

# Boosting moderno y optimización de hiperparámetros con validación correcta

**Curso:** Aprendizaje Automático — SI7009 - 1 (5553)  
**Sesión 2:** Boosting moderno y optimización de hiperparámetros con validación correcta  
**Universidad:** EAFIT  
**Dataset canónico:** Credit Card Fraud Detection  
**Profesor:** Marco Teran  

---

## Propósito del notebook

Este notebook materializa la lógica metodológica de la Sesión 2:

1. comparar familias boosted bajo condiciones homogéneas;
2. usar una métrica coherente con clasificación desbalanceada;
3. diseñar HPO con validación correcta;
4. diagnosticar Optuna sin tratarlo como magia;
5. contrastar class weights y SMOTE como hipótesis, no recetas;
6. seleccionar un threshold con evidencia out-of-fold;
7. evaluar en test reservado solo al final.

La pregunta central no es:

> ¿Qué modelo da el número más alto?

sino:

> ¿Qué pipeline merece continuar porque ganó bajo una comparación justa, con una métrica adecuada y una validación defendible?


## Tabla de contenidos

1. Contrato metodológico de la sesión
2. Setup reproducible e instalación
3. Carga robusta del dataset
4. Auditoría de datos y desbalance
5. Split train/test y validación común
6. Baselines
7. Comparación base: XGBoost, LightGBM y CatBoost
8. HPO con Optuna y fallback
9. Diagnóstico de HPO
10. Modelo tuneado bajo CV común
11. Estrategias de desbalance: none, class weights y SMOTE
12. Selección provisional por CV
13. Out-of-fold scores y threshold policy
14. Evaluación final en test reservado
15. Importancia de variables
16. Registro final del experimento
17. Auditoría final del notebook
18. Ejercicios de extensión

> Ejecuta las celdas en orden. La meta no es solo obtener resultados, sino poder defender la comparación.


## 1. Contrato metodológico de la sesión

Durante todo el notebook se aplican estas reglas:

- No se usa `accuracy` como métrica principal.
- La métrica principal será `average_precision`, equivalente operativa a PR-AUC en `scikit-learn`.
- Las comparaciones deben usar el mismo esquema de validación.
- El test set no se usa para escoger hiperparámetros, familia, estrategia de desbalance ni threshold.
- SMOTE, class weights y modelos sin intervención se tratan como estrategias comparables, no como dogmas.
- Un modelo no se declara ganador solo por el mejor promedio: también se revisa estabilidad, complejidad, costo y coherencia metodológica.

### Qué se espera al finalizar

Debes poder explicar:

- por qué boosting moderno es relevante para problemas tabulares;
- qué diferencias prácticas importan entre XGBoost, LightGBM y CatBoost;
- qué controlan `learning_rate`, profundidad, número de árboles, regularización y subsampling;
- por qué Optuna optimiza una función objetivo diseñada por el analista;
- por qué PR-AUC es más apropiado que accuracy en fraude;
- cuándo class weights o SMOTE merecen conservarse;
- qué evidencia mínima permite decir que un pipeline ganó de forma defendible.


## 2. Setup reproducible e instalación

Este notebook está diseñado para Colab, Kaggle y Jupyter local.

Librerías requeridas para el flujo base:

- `numpy`
- `pandas`
- `matplotlib`
- `scikit-learn`

Librerías recomendadas para el flujo completo:

- `xgboost`
- `lightgbm`
- `catboost`
- `optuna`
- `imbalanced-learn`

### Instalación en Colab o entorno local

Ejecuta la siguiente celda solo si alguna librería falta. Después de instalar, puede ser necesario reiniciar el runtime/kernel.

```bash
pip install -U xgboost lightgbm catboost optuna imbalanced-learn
```

Optuna puede fallar en algunos entornos por dependencias visuales, versiones antiguas de Python o conflictos con `plotly`. Si eso ocurre, el notebook usa un fallback con `RandomizedSearchCV` para preservar la lógica de HPO.


In [ ]:
# =============================================================================
# 2.1 Optional installation cell
# =============================================================================
# Uncomment and run only if needed.
# In Colab, restart the runtime after installation if imports still fail.

# !pip install -U -q xgboost lightgbm catboost optuna imbalanced-learn


In [ ]:
# =============================================================================
# 2.2 Imports and global configuration
# =============================================================================

from __future__ import annotations

import os
import sys
import time
import warnings
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

import sklearn
from packaging import version

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold,
    cross_val_predict,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

assert version.parse(sklearn.__version__) >= version.parse("1.2.0"), (
    "Este notebook requiere scikit-learn >= 1.2.0"
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

plt.rcParams["figure.figsize"] = (10, 5.5)
plt.rcParams["font.size"] = 11
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print("Setup base importado correctamente.")


In [ ]:
# =============================================================================
# 2.3 Optional package imports
# =============================================================================

XGBClassifier = None
LGBMClassifier = None
CatBoostClassifier = None
optuna = None
ImbPipeline = None
SMOTE = None

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception as exc:
    XGBOOST_AVAILABLE = False
    print(f"XGBoost no disponible: {exc}")

try:
    from lightgbm import LGBMClassifier
    LIGHTGBM_AVAILABLE = True
except Exception as exc:
    LIGHTGBM_AVAILABLE = False
    print(f"LightGBM no disponible: {exc}")

try:
    from catboost import CatBoostClassifier
    CATBOOST_AVAILABLE = True
except Exception as exc:
    CATBOOST_AVAILABLE = False
    print(f"CatBoost no disponible: {exc}")

try:
    import optuna
    OPTUNA_AVAILABLE = True
except Exception as exc:
    OPTUNA_AVAILABLE = False
    print(f"Optuna no disponible: {exc}")

try:
    from imblearn.pipeline import Pipeline as ImbPipeline
    from imblearn.over_sampling import SMOTE
    IMBLEARN_AVAILABLE = True
except Exception as exc:
    IMBLEARN_AVAILABLE = False
    print(f"imbalanced-learn no disponible: {exc}")

package_availability = pd.DataFrame([
    {"package": "xgboost", "available": XGBOOST_AVAILABLE},
    {"package": "lightgbm", "available": LIGHTGBM_AVAILABLE},
    {"package": "catboost", "available": CATBOOST_AVAILABLE},
    {"package": "optuna", "available": OPTUNA_AVAILABLE},
    {"package": "imbalanced-learn", "available": IMBLEARN_AVAILABLE},
])

display(package_availability)

if not any([XGBOOST_AVAILABLE, LIGHTGBM_AVAILABLE, CATBOOST_AVAILABLE]):
    raise ImportError(
        "Se requiere al menos una librería boosted: xgboost, lightgbm o catboost. "
        "Instala las dependencias y reinicia el runtime."
    )


In [ ]:
# =============================================================================
# 2.4 Notebook configuration
# =============================================================================

TARGET_COL = "Class"
NEGATIVE_LABEL = 0
POSITIVE_LABEL = 1

N_SPLITS = 5
TEST_SIZE = 0.20

# FAST_DEMO_MODE keeps the classroom run realistic.
# Set to False for a stronger but slower experiment.
FAST_DEMO_MODE = True

if FAST_DEMO_MODE:
    HPO_N_TRIALS = 15
    HPO_N_SPLITS = 3
    HPO_MAX_TRAIN_ROWS = 80_000
else:
    HPO_N_TRIALS = 50
    HPO_N_SPLITS = 5
    HPO_MAX_TRAIN_ROWS = None

# Avoid aggressive nested parallelism in classroom notebooks.
N_JOBS_OUTER = 1
N_JOBS_MODEL = -1

CLASSIFICATION_CV_SCORING = {
    "pr_auc": "average_precision",
    "roc_auc": "roc_auc",
    "f1": "f1",
    "precision": "precision",
    "recall": "recall",
}

model_registry: Dict[str, Any] = {}
results_registry: Dict[str, pd.DataFrame] = {}

print("Configuration ready.")
print(f"FAST_DEMO_MODE={FAST_DEMO_MODE}")
print(f"HPO_N_TRIALS={HPO_N_TRIALS}")
print(f"HPO_N_SPLITS={HPO_N_SPLITS}")


In [ ]:
# =============================================================================
# 2.5 General helper functions
# =============================================================================

def print_section(title: str, width: int = 96) -> None:
    """Print a readable section header."""
    print("\n" + "=" * width)
    print(title.center(width))
    print("=" * width)


def require_columns(
    dataframe: pd.DataFrame,
    required_columns: Sequence[str],
    dataframe_name: str = "dataframe",
) -> None:
    """Raise a clear error if required columns are missing."""
    missing = [col for col in required_columns if col not in dataframe.columns]
    if missing:
        raise ValueError(
            f"{dataframe_name} is missing required columns: {missing}. "
            f"Available columns: {list(dataframe.columns)}"
        )


def get_binary_class_counts(y_data: pd.Series) -> Dict[str, Any]:
    """Return class counts and prevalence for a binary target."""
    counts = y_data.value_counts().sort_index()
    negative_count = int(counts.get(NEGATIVE_LABEL, 0))
    positive_count = int(counts.get(POSITIVE_LABEL, 0))
    total_count = int(len(y_data))

    if total_count == 0:
        raise ValueError("Target vector is empty.")

    return {
        "total_count": total_count,
        "negative_count": negative_count,
        "positive_count": positive_count,
        "positive_rate": positive_count / total_count,
    }


def find_credit_card_dataset() -> Optional[Path]:
    """Search for creditcard.csv in common local, Colab, and Kaggle paths."""
    candidate_paths = [
        Path("creditcard.csv"),
        Path("data/creditcard.csv"),
        Path("../input/creditcardfraud/creditcard.csv"),
        Path("/kaggle/input/creditcardfraud/creditcard.csv"),
        Path("/content/creditcard.csv"),
        Path("/content/data/creditcard.csv"),
    ]

    for path in candidate_paths:
        if path.exists():
            return path

    return None


def audit_dataframe_schema(dataframe: pd.DataFrame, target_col: str) -> pd.DataFrame:
    """Create a compact schema audit table."""
    rows = []
    for col in dataframe.columns:
        rows.append({
            "column": col,
            "dtype": str(dataframe[col].dtype),
            "missing_count": int(dataframe[col].isna().sum()),
            "missing_rate": float(dataframe[col].isna().mean()),
            "is_target": col == target_col,
        })
    return pd.DataFrame(rows)


def add_stability_columns(
    results: pd.DataFrame,
    metric_mean_col: str = "pr_auc_mean",
    metric_std_col: str = "pr_auc_std",
) -> pd.DataFrame:
    """Add simple stability indicators to a result table."""
    output = results.copy()
    output["relative_variability"] = output[metric_std_col] / output[metric_mean_col].replace(0, np.nan)
    output["rank_by_mean"] = output[metric_mean_col].rank(ascending=False, method="min")
    output["rank_by_stability"] = output["relative_variability"].rank(ascending=True, method="min")
    return output


def display_metric_columns(results: pd.DataFrame) -> pd.DataFrame:
    """Return the most useful metric columns for display."""
    preferred = [
        "model_group", "model",
        "pr_auc_mean", "pr_auc_std",
        "roc_auc_mean", "roc_auc_std",
        "precision_mean", "precision_std",
        "recall_mean", "recall_std",
        "f1_mean", "f1_std",
        "relative_variability",
        "fit_time_mean", "score_time_mean", "total_time_sec",
    ]
    cols = [col for col in preferred if col in results.columns]
    return results[cols].copy()


def summarize_cv_results(
    model_name: str,
    cv_result: Dict[str, np.ndarray],
    model_group: str,
) -> pd.DataFrame:
    """Convert sklearn cross_validate output into a one-row report."""
    row: Dict[str, Any] = {
        "model_group": model_group,
        "model": model_name,
    }

    for key, values in cv_result.items():
        if key.startswith("test_"):
            metric_name = key.replace("test_", "")
            row[f"{metric_name}_mean"] = float(np.mean(values))
            row[f"{metric_name}_std"] = float(np.std(values, ddof=1))

    row["fit_time_mean"] = float(np.mean(cv_result["fit_time"]))
    row["score_time_mean"] = float(np.mean(cv_result["score_time"]))

    return pd.DataFrame([row])


def evaluate_model_cv(
    estimator: Any,
    model_name: str,
    X_data: pd.DataFrame,
    y_data: pd.Series,
    cv_strategy: StratifiedKFold,
    scoring: Dict[str, str],
    model_group: str,
    n_jobs: int = N_JOBS_OUTER,
) -> pd.DataFrame:
    """Evaluate an estimator with a shared CV strategy."""
    start_time = time.time()

    cv_result = cross_validate(
        estimator=estimator,
        X=X_data,
        y=y_data,
        cv=cv_strategy,
        scoring=scoring,
        n_jobs=n_jobs,
        return_train_score=False,
        error_score="raise",
    )

    report = summarize_cv_results(
        model_name=model_name,
        cv_result=cv_result,
        model_group=model_group,
    )
    report["total_time_sec"] = float(time.time() - start_time)
    return report


def combine_and_sort_results(
    result_tables: Sequence[pd.DataFrame],
    sort_metric: str = "pr_auc_mean",
    ascending: bool = False,
) -> pd.DataFrame:
    """Combine result tables and sort by a metric."""
    non_empty = [table for table in result_tables if table is not None and not table.empty]
    if not non_empty:
        return pd.DataFrame()
    combined = pd.concat(non_empty, ignore_index=True)
    return combined.sort_values(sort_metric, ascending=ascending).reset_index(drop=True)


def plot_cv_metric_comparison(
    results: pd.DataFrame,
    metric_mean_col: str,
    metric_std_col: str,
    title: str,
    top_n: Optional[int] = None,
) -> None:
    """Plot mean ± std for a CV metric."""
    require_columns(results, ["model", metric_mean_col, metric_std_col], "results")

    plot_df = results.sort_values(metric_mean_col, ascending=True).copy()
    if top_n is not None:
        plot_df = plot_df.tail(top_n)

    fig, ax = plt.subplots(figsize=(10, max(4.5, 0.45 * len(plot_df))))
    ax.barh(plot_df["model"], plot_df[metric_mean_col], xerr=plot_df[metric_std_col], alpha=0.85)
    ax.set_title(title)
    ax.set_xlabel(f"{metric_mean_col} ± {metric_std_col}")
    ax.set_ylabel("Modelo / pipeline")
    plt.show()


def infer_family_from_model_name(model_name: str) -> Optional[str]:
    """Infer boosted family from a model name."""
    lowered = model_name.lower()
    if "xgboost" in lowered:
        return "xgboost"
    if "lightgbm" in lowered:
        return "lightgbm"
    if "catboost" in lowered:
        return "catboost"
    return None

print("Helper functions ready.")


## 3. Carga robusta del dataset

El notebook espera el dataset `creditcard.csv` con la columna objetivo `Class`.

Escenarios soportados:

1. Kaggle: `/kaggle/input/creditcardfraud/creditcard.csv`
2. Colab: `/content/creditcard.csv` después de subirlo manualmente
3. Local: `creditcard.csv` o `data/creditcard.csv`

La clase positiva `Class = 1` representa fraude.


In [ ]:
# =============================================================================
# 3.1 Dataset loading
# =============================================================================

dataset_path = find_credit_card_dataset()

if dataset_path is None:
    print_section("DATASET NOT FOUND")
    print("No se encontró creditcard.csv automáticamente.")
    print("Coloca el archivo en la carpeta actual, en data/creditcard.csv, o súbelo a Colab.")
    raise FileNotFoundError(
        "creditcard.csv was not found. Upload or place the file and rerun the notebook."
    )

print(f"Dataset found at: {dataset_path}")

df = pd.read_csv(dataset_path)

print(f"Dataset shape: {df.shape}")
display(df.head())


In [ ]:
# =============================================================================
# 3.2 Dataset schema and target audit
# =============================================================================

if TARGET_COL not in df.columns:
    raise ValueError(
        f"Expected target column '{TARGET_COL}' was not found. "
        f"Available columns: {list(df.columns)}"
    )

if df[TARGET_COL].nunique() != 2:
    raise ValueError(
        f"Expected a binary target in '{TARGET_COL}', found {df[TARGET_COL].nunique()} unique values."
    )

schema_audit = audit_dataframe_schema(df, TARGET_COL)

display(schema_audit)

non_numeric_predictors = [
    col for col in df.drop(columns=[TARGET_COL]).columns
    if not pd.api.types.is_numeric_dtype(df[col])
]

if non_numeric_predictors:
    raise ValueError(
        "This notebook expects numeric predictors for the canonical credit-card dataset. "
        f"Non-numeric columns found: {non_numeric_predictors}"
    )

if df.isna().any().any():
    missing_cols = df.columns[df.isna().any()].tolist()
    raise ValueError(
        f"Missing values detected in columns: {missing_cols}. "
        "Handle imputation before continuing."
    )

print("Dataset schema audit passed.")


## 4. Auditoría del desbalance

Antes de entrenar modelos, verificamos la prevalencia de fraude y mostramos por qué `accuracy` sería una brújula débil.


In [ ]:
# =============================================================================
# 4.1 Class balance audit
# =============================================================================

y_full = df[TARGET_COL].astype(int)
class_stats = get_binary_class_counts(y_full)

class_balance = pd.DataFrame([
    {"class": "no_fraud", "label": NEGATIVE_LABEL, "count": class_stats["negative_count"]},
    {"class": "fraud", "label": POSITIVE_LABEL, "count": class_stats["positive_count"]},
])
class_balance["rate"] = class_balance["count"] / class_stats["total_count"]
class_balance["rate_percent"] = class_balance["rate"] * 100

display(class_balance)

majority_accuracy = 1.0 - class_stats["positive_rate"]

print(f"Fraud prevalence: {class_stats['positive_rate']:.8f} ({class_stats['positive_rate'] * 100:.4f}%)")
print(f"Accuracy of an always-no-fraud classifier ≈ {majority_accuracy:.8f}")
print("Expected fraud recall of that trivial classifier = 0.0")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(class_balance["class"], class_balance["count"], alpha=0.85)
ax.set_yscale("log")
ax.set_title("Distribución de clases — escala logarítmica")
ax.set_xlabel("Clase")
ax.set_ylabel("Número de observaciones")
plt.show()


## 5. Split train/test y validación común

Se crea un test set estratificado que queda reservado hasta el final.

La selección de modelos, HPO, comparación de estrategias y threshold se hacen usando solo el training set.


In [ ]:
# =============================================================================
# 5.1 Train/test split
# =============================================================================

X = df.drop(columns=[TARGET_COL]).copy()
y = df[TARGET_COL].astype(int).copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

split_rows = []
for split_name, split_y in {"train": y_train, "test": y_test}.items():
    stats = get_binary_class_counts(split_y)
    split_rows.append({
        "split": split_name,
        **stats,
        "positive_rate_percent": stats["positive_rate"] * 100,
    })

split_summary = pd.DataFrame(split_rows)
display(split_summary)

if set(X_train.index).intersection(set(X_test.index)):
    raise RuntimeError("Train and test indices overlap. Review split logic.")


In [ ]:
# =============================================================================
# 5.2 Shared cross-validation strategy
# =============================================================================

cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

print(cv)
print(CLASSIFICATION_CV_SCORING)


## 6. Baselines

Antes de comparar boosting, se construyen dos referencias:

1. `DummyClassifier`: muestra la trampa de una referencia trivial.
2. `LogisticRegression` con `class_weight="balanced"`: baseline clásico, simple y defendible.

Un modelo boosted debe superar una referencia seria, no solo un modelo trivial.


In [ ]:
# =============================================================================
# 6.1 Baseline models
# =============================================================================

negative_count = int((y_train == NEGATIVE_LABEL).sum())
positive_count = int((y_train == POSITIVE_LABEL).sum())

if positive_count == 0:
    raise ValueError("No positive examples in training set. Stratified split failed.")

scale_pos_weight = negative_count / positive_count

print(f"Training negative count: {negative_count}")
print(f"Training positive count: {positive_count}")
print(f"scale_pos_weight: {scale_pos_weight:.4f}")

baseline_models = {
    "dummy_most_frequent": DummyClassifier(strategy="most_frequent"),
    "logistic_class_weight_balanced": Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            class_weight="balanced",
            max_iter=2_000,
            random_state=RANDOM_STATE,
            n_jobs=N_JOBS_MODEL,
        )),
    ]),
}

baseline_reports = []

for name, estimator in baseline_models.items():
    print_section(f"Evaluating baseline: {name}")
    report = evaluate_model_cv(
        estimator=estimator,
        model_name=name,
        X_data=X_train,
        y_data=y_train,
        cv_strategy=cv,
        scoring=CLASSIFICATION_CV_SCORING,
        model_group="baseline",
        n_jobs=N_JOBS_OUTER,
    )
    baseline_reports.append(report)
    display(display_metric_columns(report))

baseline_results = combine_and_sort_results(baseline_reports, "pr_auc_mean", ascending=False)
baseline_results = add_stability_columns(baseline_results)

model_registry.update(baseline_models)
results_registry["baseline_results"] = baseline_results

display(display_metric_columns(baseline_results))


## 7. Comparación base de familias boosted

Comparamos XGBoost, LightGBM y CatBoost cuando estén disponibles.

Reglas:

- mismo `X_train`;
- mismo `StratifiedKFold`;
- misma métrica principal;
- mismo tratamiento inicial de desbalance mediante class weights o equivalente;
- test set intacto.


In [ ]:
# =============================================================================
# 7.1 Base boosted model factory
# =============================================================================

def build_base_boosting_models(
    scale_pos_weight_value: float,
    random_state: int = RANDOM_STATE,
) -> Dict[str, Any]:
    """Build initial boosted models with comparable moderate settings."""
    models: Dict[str, Any] = {}

    if XGBOOST_AVAILABLE and XGBClassifier is not None:
        models["xgboost_base_class_weight"] = XGBClassifier(
            n_estimators=250,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.90,
            colsample_bytree=0.90,
            reg_lambda=1.0,
            objective="binary:logistic",
            eval_metric="aucpr",
            scale_pos_weight=scale_pos_weight_value,
            random_state=random_state,
            n_jobs=N_JOBS_MODEL,
            tree_method="hist",
        )

    if LIGHTGBM_AVAILABLE and LGBMClassifier is not None:
        models["lightgbm_base_class_weight"] = LGBMClassifier(
            n_estimators=250,
            learning_rate=0.05,
            num_leaves=31,
            max_depth=-1,
            min_child_samples=40,
            subsample=0.90,
            subsample_freq=1,
            colsample_bytree=0.90,
            reg_lambda=1.0,
            objective="binary",
            class_weight="balanced",
            random_state=random_state,
            n_jobs=N_JOBS_MODEL,
            verbosity=-1,
        )

    if CATBOOST_AVAILABLE and CatBoostClassifier is not None:
        models["catboost_base_class_weight"] = CatBoostClassifier(
            iterations=250,
            depth=5,
            learning_rate=0.05,
            l2_leaf_reg=3.0,
            loss_function="Logloss",
            eval_metric="PRAUC",
            class_weights=[1.0, scale_pos_weight_value],
            random_seed=random_state,
            verbose=False,
            allow_writing_files=False,
            thread_count=N_JOBS_MODEL,
        )

    return models


boosting_models = build_base_boosting_models(
    scale_pos_weight_value=scale_pos_weight,
    random_state=RANDOM_STATE,
)

if not boosting_models:
    raise ImportError("No boosted model libraries are available.")

model_registry.update(boosting_models)

print("Available boosted models:")
for name in boosting_models:
    print(f"- {name}")


In [ ]:
# =============================================================================
# 7.2 Evaluate base boosted families
# =============================================================================

boosting_reports = []

for name, estimator in boosting_models.items():
    print_section(f"Evaluating boosted model: {name}")
    report = evaluate_model_cv(
        estimator=estimator,
        model_name=name,
        X_data=X_train,
        y_data=y_train,
        cv_strategy=cv,
        scoring=CLASSIFICATION_CV_SCORING,
        model_group="boosting_base",
        n_jobs=N_JOBS_OUTER,
    )
    boosting_reports.append(report)
    display(display_metric_columns(report))

boosting_base_results = combine_and_sort_results(boosting_reports, "pr_auc_mean", ascending=False)
boosting_base_results = add_stability_columns(boosting_base_results)

initial_comparison = combine_and_sort_results(
    [baseline_results, boosting_base_results],
    sort_metric="pr_auc_mean",
    ascending=False,
)
initial_comparison = add_stability_columns(initial_comparison)

results_registry["boosting_base_results"] = boosting_base_results
results_registry["initial_comparison"] = initial_comparison

display(display_metric_columns(initial_comparison))

plot_cv_metric_comparison(
    results=initial_comparison,
    metric_mean_col="pr_auc_mean",
    metric_std_col="pr_auc_std",
    title="Comparación inicial: baselines y boosting base",
    top_n=10,
)


In [ ]:
# =============================================================================
# 7.3 Choose HPO family
# =============================================================================

def choose_hpo_family() -> str:
    """Choose a family for HPO demonstration based on availability."""
    if XGBOOST_AVAILABLE:
        return "xgboost"
    if LIGHTGBM_AVAILABLE:
        return "lightgbm"
    if CATBOOST_AVAILABLE:
        return "catboost"
    raise ImportError("No supported boosted model family is available.")


HPO_FAMILY = choose_hpo_family()

print(f"Selected HPO family: {HPO_FAMILY}")
print(
    "Esta selección es docente y práctica; no significa que esa familia gane siempre. "
    "Lo importante es preservar la metodología de HPO."
)


## 8. HPO con Optuna y fallback

Optuna no reemplaza el criterio metodológico. Solo optimiza la función objetivo que definimos.

La función objetivo debe incluir:

- métrica coherente: `average_precision`;
- validación correcta: `StratifiedKFold`;
- espacio de búsqueda razonable;
- ausencia total de test set;
- lectura de promedio y variabilidad.


In [ ]:
# =============================================================================
# 8.1 HPO sampling and model-building helpers
# =============================================================================

hpo_cv = StratifiedKFold(
    n_splits=HPO_N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)


def make_stratified_sample(
    X_data: pd.DataFrame,
    y_data: pd.Series,
    max_rows: Optional[int],
    random_state: int = RANDOM_STATE,
) -> Tuple[pd.DataFrame, pd.Series]:
    """Create a stratified sample for faster HPO demonstrations."""
    if max_rows is None or len(X_data) <= max_rows:
        return X_data.copy(), y_data.copy()
    if max_rows <= 0:
        raise ValueError("max_rows must be positive or None.")

    train_size = max_rows / len(X_data)
    X_sample, _, y_sample, _ = train_test_split(
        X_data,
        y_data,
        train_size=train_size,
        stratify=y_data,
        random_state=random_state,
    )
    return X_sample.copy(), y_sample.copy()


X_hpo, y_hpo = make_stratified_sample(
    X_data=X_train,
    y_data=y_train,
    max_rows=HPO_MAX_TRAIN_ROWS,
    random_state=RANDOM_STATE,
)

hpo_stats = get_binary_class_counts(y_hpo)
hpo_scale_pos_weight = hpo_stats["negative_count"] / hpo_stats["positive_count"]

hpo_sample_summary = pd.DataFrame([
    {"dataset": "full_training_set", **get_binary_class_counts(y_train)},
    {"dataset": "hpo_training_sample", **hpo_stats},
])
display(hpo_sample_summary)

print(f"HPO scale_pos_weight: {hpo_scale_pos_weight:.4f}")


def sample_xgboost_params(trial: Any) -> Dict[str, Any]:
    """Sample a compact XGBoost search space."""
    return {
        "n_estimators": trial.suggest_int("n_estimators", 150, 650),
        "max_depth": trial.suggest_int("max_depth", 2, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.20, log=True),
        "subsample": trial.suggest_float("subsample", 0.65, 1.00),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.65, 1.00),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 12.0),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 20.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 5.0, log=True),
    }


def sample_lightgbm_params(trial: Any) -> Dict[str, Any]:
    """Sample a compact LightGBM search space."""
    return {
        "n_estimators": trial.suggest_int("n_estimators", 150, 650),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.20, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 15, 127),
        "max_depth": trial.suggest_int("max_depth", -1, 10),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 120),
        "subsample": trial.suggest_float("subsample", 0.65, 1.00),
        "subsample_freq": trial.suggest_int("subsample_freq", 1, 5),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.65, 1.00),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 20.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 5.0, log=True),
    }


def sample_catboost_params(trial: Any) -> Dict[str, Any]:
    """Sample a compact CatBoost search space."""
    return {
        "iterations": trial.suggest_int("iterations", 150, 650),
        "depth": trial.suggest_int("depth", 3, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.20, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 20.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 0.1, 5.0),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 5.0),
    }


def sample_params_for_family(trial: Any, family: str) -> Dict[str, Any]:
    """Route Optuna sampling to the selected model family."""
    if family == "xgboost":
        return sample_xgboost_params(trial)
    if family == "lightgbm":
        return sample_lightgbm_params(trial)
    if family == "catboost":
        return sample_catboost_params(trial)
    raise ValueError(f"Unsupported family: {family}")


def get_default_hpo_params_for_family(family: str) -> Dict[str, Any]:
    """Return compact defaults compatible with build_trial_model."""
    if family == "xgboost":
        return {
            "n_estimators": 250,
            "max_depth": 4,
            "learning_rate": 0.05,
            "subsample": 0.90,
            "colsample_bytree": 0.90,
            "min_child_weight": 1.0,
            "gamma": 0.0,
            "reg_lambda": 1.0,
            "reg_alpha": 0.0,
        }
    if family == "lightgbm":
        return {
            "n_estimators": 250,
            "learning_rate": 0.05,
            "num_leaves": 31,
            "max_depth": -1,
            "min_child_samples": 40,
            "subsample": 0.90,
            "subsample_freq": 1,
            "colsample_bytree": 0.90,
            "reg_lambda": 1.0,
            "reg_alpha": 0.0,
        }
    if family == "catboost":
        return {
            "iterations": 250,
            "depth": 5,
            "learning_rate": 0.05,
            "l2_leaf_reg": 3.0,
            "random_strength": 1.0,
            "bagging_temperature": 1.0,
        }
    raise ValueError(f"Unsupported family: {family}")


def build_trial_model(
    family: str,
    params: Dict[str, Any],
    scale_pos_weight_value: float,
    random_state: int = RANDOM_STATE,
    n_jobs_model: int = N_JOBS_MODEL,
) -> Any:
    """Build a boosted model for a single HPO configuration."""
    if family == "xgboost":
        if not XGBOOST_AVAILABLE or XGBClassifier is None:
            raise ImportError("XGBoost is not available.")
        return XGBClassifier(
            **params,
            objective="binary:logistic",
            eval_metric="aucpr",
            scale_pos_weight=scale_pos_weight_value,
            random_state=random_state,
            n_jobs=n_jobs_model,
            tree_method="hist",
        )

    if family == "lightgbm":
        if not LIGHTGBM_AVAILABLE or LGBMClassifier is None:
            raise ImportError("LightGBM is not available.")
        return LGBMClassifier(
            **params,
            objective="binary",
            class_weight="balanced",
            random_state=random_state,
            n_jobs=n_jobs_model,
            verbosity=-1,
        )

    if family == "catboost":
        if not CATBOOST_AVAILABLE or CatBoostClassifier is None:
            raise ImportError("CatBoost is not available.")
        return CatBoostClassifier(
            **params,
            loss_function="Logloss",
            eval_metric="PRAUC",
            class_weights=[1.0, scale_pos_weight_value],
            random_seed=random_state,
            verbose=False,
            allow_writing_files=False,
            thread_count=n_jobs_model,
        )

    raise ValueError(f"Unsupported family: {family}")


In [ ]:
# =============================================================================
# 8.2 Optuna objective and fallback search
# =============================================================================

def objective(trial: Any) -> float:
    """Optuna objective: mean average precision under Stratified K-Fold."""
    params = sample_params_for_family(trial=trial, family=HPO_FAMILY)
    estimator = build_trial_model(
        family=HPO_FAMILY,
        params=params,
        scale_pos_weight_value=hpo_scale_pos_weight,
        random_state=RANDOM_STATE,
        n_jobs_model=N_JOBS_MODEL,
    )

    cv_result = cross_validate(
        estimator=estimator,
        X=X_hpo,
        y=y_hpo,
        cv=hpo_cv,
        scoring={"pr_auc": "average_precision"},
        n_jobs=N_JOBS_OUTER,
        return_train_score=False,
        error_score="raise",
    )

    fold_scores = cv_result["test_pr_auc"]
    trial.set_user_attr("fold_scores", fold_scores.tolist())
    trial.set_user_attr("std_pr_auc", float(np.std(fold_scores, ddof=1)))
    trial.set_user_attr("fit_time_mean", float(np.mean(cv_result["fit_time"])))

    return float(np.mean(fold_scores))


def get_randomized_search_space_for_family(family: str) -> Dict[str, List[Any]]:
    """Return a compact discrete search space for fallback HPO."""
    if family == "xgboost":
        return {
            "n_estimators": [150, 250, 400, 600],
            "max_depth": [2, 3, 4, 5, 6, 8],
            "learning_rate": [0.01, 0.03, 0.05, 0.10, 0.20],
            "subsample": [0.70, 0.85, 1.00],
            "colsample_bytree": [0.70, 0.85, 1.00],
            "min_child_weight": [1.0, 3.0, 6.0, 10.0],
            "gamma": [0.0, 1.0, 3.0, 5.0],
            "reg_lambda": [0.1, 1.0, 5.0, 10.0],
            "reg_alpha": [0.0, 0.01, 0.1, 1.0],
        }
    if family == "lightgbm":
        return {
            "n_estimators": [150, 250, 400, 600],
            "learning_rate": [0.01, 0.03, 0.05, 0.10, 0.20],
            "num_leaves": [15, 31, 63, 127],
            "max_depth": [-1, 3, 5, 7, 10],
            "min_child_samples": [10, 20, 40, 80, 120],
            "subsample": [0.70, 0.85, 1.00],
            "subsample_freq": [1, 3, 5],
            "colsample_bytree": [0.70, 0.85, 1.00],
            "reg_lambda": [0.1, 1.0, 5.0, 10.0],
            "reg_alpha": [0.0, 0.01, 0.1, 1.0],
        }
    if family == "catboost":
        return {
            "iterations": [150, 250, 400, 600],
            "depth": [3, 4, 5, 6, 8],
            "learning_rate": [0.01, 0.03, 0.05, 0.10, 0.20],
            "l2_leaf_reg": [1.0, 3.0, 5.0, 10.0, 20.0],
            "random_strength": [0.1, 1.0, 3.0, 5.0],
            "bagging_temperature": [0.0, 1.0, 3.0, 5.0],
        }
    raise ValueError(f"Unsupported family: {family}")


HPO_COMPLETED = False
HPO_USED_FALLBACK = False
HPO_FAILURE_REASON = None
study = None
hpo_search_object = None
best_hpo_params: Dict[str, Any] = {}
best_hpo_value: Optional[float] = None
hpo_trials_report = pd.DataFrame()

if OPTUNA_AVAILABLE and optuna is not None:
    print_section("RUNNING OPTUNA HPO")
    try:
        optuna.logging.set_verbosity(optuna.logging.WARNING)
        study = optuna.create_study(
            direction="maximize",
            study_name=f"{HPO_FAMILY}_average_precision_hpo",
            sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
        )
        hpo_start_time = time.time()
        study.optimize(objective, n_trials=HPO_N_TRIALS, show_progress_bar=True)
        hpo_elapsed_time = time.time() - hpo_start_time

        best_hpo_params = dict(study.best_params)
        best_hpo_value = float(study.best_value)

        trials_df = study.trials_dataframe()
        preferred = ["number", "value", "user_attrs_std_pr_auc", "user_attrs_fit_time_mean", "duration", "state"]
        param_cols = [col for col in trials_df.columns if col.startswith("params_")]
        hpo_trials_report = trials_df[[col for col in preferred if col in trials_df.columns] + param_cols]
        hpo_trials_report = hpo_trials_report.sort_values("value", ascending=False)

        HPO_COMPLETED = True
        print(f"Optuna HPO completed in {hpo_elapsed_time:.2f} seconds.")
        print(f"Best mean PR-AUC: {best_hpo_value:.6f}")
    except Exception as exc:
        HPO_FAILURE_REASON = str(exc)
        print("Optuna failed. Activating fallback.")
        print(HPO_FAILURE_REASON)

if not HPO_COMPLETED:
    print_section("RUNNING RANDOMIZEDSEARCHCV FALLBACK")
    fallback_estimator = build_trial_model(
        family=HPO_FAMILY,
        params=get_default_hpo_params_for_family(HPO_FAMILY),
        scale_pos_weight_value=hpo_scale_pos_weight,
        random_state=RANDOM_STATE,
        n_jobs_model=N_JOBS_MODEL,
    )
    hpo_search_object = RandomizedSearchCV(
        estimator=fallback_estimator,
        param_distributions=get_randomized_search_space_for_family(HPO_FAMILY),
        n_iter=min(HPO_N_TRIALS, 20),
        scoring="average_precision",
        cv=hpo_cv,
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS_OUTER,
        refit=False,
        verbose=1,
        error_score="raise",
    )
    fallback_start = time.time()
    hpo_search_object.fit(X_hpo, y_hpo)
    fallback_elapsed = time.time() - fallback_start

    best_hpo_params = dict(hpo_search_object.best_params_)
    best_hpo_value = float(hpo_search_object.best_score_)
    hpo_trials_report = pd.DataFrame(hpo_search_object.cv_results_).sort_values(
        "mean_test_score",
        ascending=False,
    )
    HPO_COMPLETED = True
    HPO_USED_FALLBACK = True
    print(f"Fallback HPO completed in {fallback_elapsed:.2f} seconds.")
    print(f"Best mean PR-AUC: {best_hpo_value:.6f}")

if not HPO_COMPLETED:
    raise RuntimeError("HPO did not complete.")

print("Best HPO parameters:")
for key, value in best_hpo_params.items():
    print(f"- {key}: {value}")

display(hpo_trials_report.head(10))


## 9. Diagnóstico de HPO

El mejor trial no reemplaza una lectura crítica. Se revisa la evolución de la búsqueda, la distribución de scores y la importancia aproximada de hiperparámetros cuando Optuna lo permite.


In [ ]:
# =============================================================================
# 9.1 HPO diagnostics
# =============================================================================

def normalize_hpo_trials_report(trials_report: pd.DataFrame, used_fallback: bool) -> pd.DataFrame:
    """Normalize Optuna or RandomizedSearchCV reports to a common schema."""
    if trials_report.empty:
        raise ValueError("HPO trials report is empty.")
    normalized = trials_report.copy().reset_index(drop=True)
    if used_fallback:
        normalized["trial_number"] = np.arange(len(normalized))
        normalized["score"] = normalized["mean_test_score"]
        normalized["score_std"] = normalized.get("std_test_score", np.nan)
    else:
        normalized["trial_number"] = normalized["number"].astype(int)
        normalized["score"] = normalized["value"].astype(float)
        normalized["score_std"] = normalized.get("user_attrs_std_pr_auc", np.nan)
    ordered = normalized.sort_values("trial_number").copy()
    ordered["best_so_far"] = ordered["score"].cummax()
    return ordered


normalized_hpo_report = normalize_hpo_trials_report(hpo_trials_report, HPO_USED_FALLBACK)

display(normalized_hpo_report.head(10))

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.scatter(normalized_hpo_report["trial_number"], normalized_hpo_report["score"], alpha=0.75, label="Score del trial")
ax.plot(normalized_hpo_report["trial_number"], normalized_hpo_report["best_so_far"], linewidth=2, label="Mejor acumulado")
ax.set_title(f"HPO de {HPO_FAMILY}: evolución de PR-AUC")
ax.set_xlabel("Trial")
ax.set_ylabel("PR-AUC promedio en validación")
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(normalized_hpo_report["score"].dropna(), bins=min(20, max(5, len(normalized_hpo_report)//2)), edgecolor="black", alpha=0.8)
ax.axvline(normalized_hpo_report["score"].max(), linestyle="--", linewidth=2, label="Mejor score")
ax.set_title("Distribución de scores durante HPO")
ax.set_xlabel("PR-AUC promedio")
ax.set_ylabel("Frecuencia")
ax.legend()
plt.show()

parameter_importance_df = pd.DataFrame()

if OPTUNA_AVAILABLE and not HPO_USED_FALLBACK and study is not None:
    try:
        from optuna.importance import get_param_importances
        importance = get_param_importances(study)
        parameter_importance_df = pd.DataFrame({
            "parameter": list(importance.keys()),
            "importance": list(importance.values()),
        }).sort_values("importance", ascending=False)
        display(parameter_importance_df)

        fig, ax = plt.subplots(figsize=(9, 5))
        plot_df = parameter_importance_df.sort_values("importance")
        ax.barh(plot_df["parameter"], plot_df["importance"])
        ax.set_title("Importancia aproximada de hiperparámetros según Optuna")
        ax.set_xlabel("Importancia relativa")
        plt.show()
    except Exception as exc:
        print(f"Could not compute Optuna parameter importances: {exc}")
else:
    print("Parameter importance via Optuna is not available in fallback mode.")

results_registry["normalized_hpo_report"] = normalized_hpo_report
results_registry["hpo_trials_report"] = hpo_trials_report


In [ ]:
# =============================================================================
# 9.2 Interpret HPO parameters and build tuned model
# =============================================================================

def interpret_hpo_parameter(parameter_name: str, value: Any) -> str:
    """Provide a short pedagogical interpretation of a hyperparameter."""
    if parameter_name == "learning_rate":
        return "Controla el tamaño de cada corrección agregada al ensemble."
    if parameter_name in {"n_estimators", "iterations"}:
        return "Controla cuántas correcciones secuenciales se acumulan."
    if parameter_name in {"max_depth", "depth"}:
        return "Controla la complejidad local de cada árbol."
    if parameter_name == "num_leaves":
        return "Controla la complejidad leaf-wise en LightGBM."
    if parameter_name in {"min_child_weight", "min_child_samples"}:
        return "Exige soporte mínimo para divisiones u hojas."
    if parameter_name == "gamma":
        return "Penaliza splits que no aportan suficiente ganancia."
    if parameter_name in {"reg_lambda", "reg_alpha", "l2_leaf_reg"}:
        return "Introduce regularización para reducir soluciones frágiles."
    if parameter_name == "subsample":
        return "Controla la fracción de filas usada por iteración."
    if parameter_name == "colsample_bytree":
        return "Controla la fracción de variables usada por árbol."
    if parameter_name in {"random_strength", "bagging_temperature", "subsample_freq"}:
        return "Controla aleatoriedad o regularización estocástica específica de la librería."
    return "Hiperparámetro específico de la familia seleccionada."


best_params_interpretation = pd.DataFrame([
    {"parameter": key, "value": value, "interpretation": interpret_hpo_parameter(key, value)}
    for key, value in best_hpo_params.items()
])

display(best_params_interpretation)

best_hpo_model = build_trial_model(
    family=HPO_FAMILY,
    params=best_hpo_params,
    scale_pos_weight_value=scale_pos_weight,
    random_state=RANDOM_STATE,
    n_jobs_model=N_JOBS_MODEL,
)

best_hpo_model_name = f"{HPO_FAMILY}_hpo_tuned"
model_registry[best_hpo_model_name] = best_hpo_model
print(best_hpo_model_name)
print(best_hpo_model)


## 10. Modelo tuneado bajo CV común

El HPO pudo haberse hecho con menos folds o submuestra por razones docentes. Por eso, el modelo tuneado vuelve a competir bajo el CV común del notebook.


In [ ]:
# =============================================================================
# 10.1 Evaluate tuned model under shared CV
# =============================================================================

print_section(f"Evaluating tuned model: {best_hpo_model_name}")

tuned_report = evaluate_model_cv(
    estimator=best_hpo_model,
    model_name=best_hpo_model_name,
    X_data=X_train,
    y_data=y_train,
    cv_strategy=cv,
    scoring=CLASSIFICATION_CV_SCORING,
    model_group="boosting_tuned",
    n_jobs=N_JOBS_OUTER,
)

tuned_report = add_stability_columns(tuned_report)

display(display_metric_columns(tuned_report))

comparison_with_tuned = combine_and_sort_results(
    [initial_comparison, tuned_report],
    "pr_auc_mean",
    ascending=False,
).drop_duplicates(subset=["model"], keep="last").reset_index(drop=True)
comparison_with_tuned = add_stability_columns(comparison_with_tuned)

display(display_metric_columns(comparison_with_tuned))

plot_cv_metric_comparison(
    comparison_with_tuned,
    "pr_auc_mean",
    "pr_auc_std",
    "Comparación inicial + modelo tuneado",
    top_n=10,
)

results_registry["tuned_report"] = tuned_report
results_registry["comparison_with_tuned"] = comparison_with_tuned


## 11. Estrategias de desbalance: none, class weights y SMOTE seguro

Comparamos tres hipótesis:

1. **Sin intervención adicional:** el modelo aprende sobre la distribución original.
2. **Class weights:** la clase positiva recibe mayor peso en la función de pérdida.
3. **SMOTE dentro del pipeline:** se generan ejemplos sintéticos solo dentro del bloque de entrenamiento de cada fold.

SMOTE nunca se aplica antes de cross-validation.


In [ ]:
# =============================================================================
# 11.1 Imbalance strategy builders
# =============================================================================

def build_model_without_extra_imbalance(
    family: str,
    params: Optional[Dict[str, Any]] = None,
    random_state: int = RANDOM_STATE,
    n_jobs_model: int = N_JOBS_MODEL,
) -> Any:
    """Build a boosted model without class weights or resampling."""
    params = dict(params or get_default_hpo_params_for_family(family))

    if family == "xgboost":
        return XGBClassifier(
            **params,
            objective="binary:logistic",
            eval_metric="aucpr",
            random_state=random_state,
            n_jobs=n_jobs_model,
            tree_method="hist",
        )
    if family == "lightgbm":
        return LGBMClassifier(
            **params,
            objective="binary",
            random_state=random_state,
            n_jobs=n_jobs_model,
            verbosity=-1,
        )
    if family == "catboost":
        return CatBoostClassifier(
            **params,
            loss_function="Logloss",
            eval_metric="PRAUC",
            random_seed=random_state,
            verbose=False,
            allow_writing_files=False,
            thread_count=n_jobs_model,
        )
    raise ValueError(f"Unsupported family: {family}")


def build_model_with_class_weights(
    family: str,
    params: Optional[Dict[str, Any]] = None,
    scale_pos_weight_value: float = scale_pos_weight,
    random_state: int = RANDOM_STATE,
    n_jobs_model: int = N_JOBS_MODEL,
) -> Any:
    """Build a boosted model with class-weight logic."""
    return build_trial_model(
        family=family,
        params=dict(params or get_default_hpo_params_for_family(family)),
        scale_pos_weight_value=scale_pos_weight_value,
        random_state=random_state,
        n_jobs_model=n_jobs_model,
    )


def build_safe_smote_pipeline(
    family: str,
    params: Optional[Dict[str, Any]] = None,
    sampling_strategy: float = 0.10,
    random_state: int = RANDOM_STATE,
    n_jobs_model: int = N_JOBS_MODEL,
) -> Any:
    """Build StandardScaler -> SMOTE -> boosted model pipeline."""
    if not IMBLEARN_AVAILABLE or ImbPipeline is None or SMOTE is None:
        raise ImportError("imbalanced-learn is not available. Install it with pip install imbalanced-learn")

    model = build_model_without_extra_imbalance(
        family=family,
        params=params,
        random_state=random_state,
        n_jobs_model=n_jobs_model,
    )

    return ImbPipeline(steps=[
        ("scaler", StandardScaler()),
        ("smote", SMOTE(sampling_strategy=sampling_strategy, k_neighbors=5, random_state=random_state)),
        ("model", model),
    ])


base_params_for_strategy = get_default_hpo_params_for_family(HPO_FAMILY)
tuned_params_for_strategy = dict(best_hpo_params)

strategy_models: Dict[str, Any] = {}
strategy_models[f"{HPO_FAMILY}_base_no_extra_imbalance"] = build_model_without_extra_imbalance(HPO_FAMILY, base_params_for_strategy)
strategy_models[f"{HPO_FAMILY}_base_class_weights"] = build_model_with_class_weights(HPO_FAMILY, base_params_for_strategy, scale_pos_weight)
strategy_models[f"{HPO_FAMILY}_tuned_class_weights"] = build_model_with_class_weights(HPO_FAMILY, tuned_params_for_strategy, scale_pos_weight)

if IMBLEARN_AVAILABLE:
    strategy_models[f"{HPO_FAMILY}_base_smote_pipeline"] = build_safe_smote_pipeline(HPO_FAMILY, base_params_for_strategy, sampling_strategy=0.10)
    strategy_models[f"{HPO_FAMILY}_tuned_smote_pipeline"] = build_safe_smote_pipeline(HPO_FAMILY, tuned_params_for_strategy, sampling_strategy=0.10)
else:
    print("SMOTE strategies skipped because imbalanced-learn is unavailable.")

model_registry.update(strategy_models)

strategy_audit = pd.DataFrame([
    {
        "strategy": name,
        "uses_tuned_params": "tuned" in name,
        "uses_class_weight": "class_weights" in name,
        "uses_smote": "smote" in name,
        "test_set_used": False,
    }
    for name in strategy_models
])

display(strategy_audit)


In [ ]:
# =============================================================================
# 11.2 Evaluate imbalance strategies
# =============================================================================

strategy_reports = []

for name, estimator in strategy_models.items():
    print_section(f"Evaluating strategy: {name}")
    report = evaluate_model_cv(
        estimator=estimator,
        model_name=name,
        X_data=X_train,
        y_data=y_train,
        cv_strategy=cv,
        scoring=CLASSIFICATION_CV_SCORING,
        model_group="imbalance_strategy",
        n_jobs=N_JOBS_OUTER,
    )
    strategy_reports.append(report)
    display(display_metric_columns(report))

strategy_results = combine_and_sort_results(strategy_reports, "pr_auc_mean", ascending=False)
strategy_results = add_stability_columns(strategy_results)

results_registry["strategy_results"] = strategy_results

display(display_metric_columns(strategy_results))

plot_cv_metric_comparison(
    strategy_results,
    "pr_auc_mean",
    "pr_auc_std",
    f"Estrategias de desbalance con {HPO_FAMILY}",
)


## 12. Selección provisional por CV

La selección provisional usa únicamente evidencia de validación cruzada. El test set sigue intacto.


In [ ]:
# =============================================================================
# 12.1 Consolidated CV comparison and candidate selection
# =============================================================================

final_cv_comparison = combine_and_sort_results(
    [baseline_results, boosting_base_results, tuned_report, strategy_results],
    sort_metric="pr_auc_mean",
    ascending=False,
).drop_duplicates(subset=["model"], keep="last").reset_index(drop=True)

final_cv_comparison = add_stability_columns(final_cv_comparison)
results_registry["final_cv_comparison"] = final_cv_comparison

display(display_metric_columns(final_cv_comparison))

plot_cv_metric_comparison(
    final_cv_comparison,
    "pr_auc_mean",
    "pr_auc_std",
    "Comparación CV consolidada antes de threshold y test",
    top_n=12,
)

candidate_cv_row = final_cv_comparison.iloc[0]
CANDIDATE_MODEL_NAME = str(candidate_cv_row["model"])
CANDIDATE_MODEL_GROUP = str(candidate_cv_row["model_group"])
CANDIDATE_PR_AUC_MEAN = float(candidate_cv_row["pr_auc_mean"])
CANDIDATE_PR_AUC_STD = float(candidate_cv_row["pr_auc_std"])

if CANDIDATE_MODEL_NAME not in model_registry:
    raise ValueError(f"Candidate model '{CANDIDATE_MODEL_NAME}' not found in model_registry.")

candidate_model = model_registry[CANDIDATE_MODEL_NAME]

candidate_selection_summary = pd.DataFrame([{
    "candidate_model": CANDIDATE_MODEL_NAME,
    "candidate_group": CANDIDATE_MODEL_GROUP,
    "cv_pr_auc_mean": CANDIDATE_PR_AUC_MEAN,
    "cv_pr_auc_std": CANDIDATE_PR_AUC_STD,
    "test_set_used": False,
}])

display(candidate_selection_summary)
print(candidate_model)


## 13. Out-of-fold scores y threshold policy

El candidato produce scores. El threshold convierte scores en decisiones.

Usaremos out-of-fold predictions sobre training para seleccionar una política sin tocar test.


In [ ]:
# =============================================================================
# 13.1 OOF scores and curves
# =============================================================================

def get_out_of_fold_scores(
    estimator: Any,
    X_data: pd.DataFrame,
    y_data: pd.Series,
    cv_strategy: StratifiedKFold,
    n_jobs: int = N_JOBS_OUTER,
) -> np.ndarray:
    """Generate out-of-fold positive-class scores."""
    try:
        proba = cross_val_predict(
            estimator=estimator,
            X=X_data,
            y=y_data,
            cv=cv_strategy,
            method="predict_proba",
            n_jobs=n_jobs,
        )
        scores = proba[:, 1]
    except Exception as proba_exc:
        print(f"predict_proba failed, using decision_function. Reason: {proba_exc}")
        raw_scores = cross_val_predict(
            estimator=estimator,
            X=X_data,
            y=y_data,
            cv=cv_strategy,
            method="decision_function",
            n_jobs=n_jobs,
        )
        raw_scores = np.asarray(raw_scores, dtype=float)
        scores = (raw_scores - raw_scores.min()) / (raw_scores.max() - raw_scores.min())
    scores = np.asarray(scores, dtype=float)
    if np.isnan(scores).any():
        raise ValueError("OOF scores contain NaN values.")
    return scores


print_section(f"OOF SCORES FOR CANDIDATE: {CANDIDATE_MODEL_NAME}")
oof_scores = get_out_of_fold_scores(candidate_model, X_train, y_train, cv, n_jobs=N_JOBS_OUTER)

OOF_PR_AUC = float(average_precision_score(y_train, oof_scores))
OOF_ROC_AUC = float(roc_auc_score(y_train, oof_scores))

oof_score_summary = pd.DataFrame([{
    "candidate_model": CANDIDATE_MODEL_NAME,
    "n_oof_scores": len(oof_scores),
    "min_score": float(np.min(oof_scores)),
    "median_score": float(np.median(oof_scores)),
    "max_score": float(np.max(oof_scores)),
    "oof_pr_auc": OOF_PR_AUC,
    "oof_roc_auc": OOF_ROC_AUC,
    "test_set_used": False,
}])

display(oof_score_summary)

precision_curve, recall_curve, pr_thresholds = precision_recall_curve(y_train, oof_scores)
fpr_curve, tpr_curve, roc_thresholds = roc_curve(y_train, oof_scores)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(recall_curve, precision_curve, linewidth=2, label=f"OOF PR, AP={OOF_PR_AUC:.4f}")
ax.axhline(float(y_train.mean()), linestyle="--", linewidth=1.5, label="Prevalencia")
ax.set_title("Curva Precision-Recall out-of-fold")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr_curve, tpr_curve, linewidth=2, label=f"OOF ROC, AUC={OOF_ROC_AUC:.4f}")
ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1.5, label="Referencia aleatoria")
ax.set_title("Curva ROC out-of-fold")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend()
plt.show()

score_df = pd.DataFrame({"true_label": y_train.values, "score": oof_scores})
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.hist(score_df.query("true_label == 0")["score"], bins=50, alpha=0.70, label="No fraude")
ax.hist(score_df.query("true_label == 1")["score"], bins=50, alpha=0.70, label="Fraude")
ax.set_yscale("log")
ax.set_title("Distribución de scores OOF por clase")
ax.set_xlabel("Score")
ax.set_ylabel("Frecuencia")
ax.legend()
plt.show()


In [ ]:
# =============================================================================
# 13.2 Threshold sweep and policy
# =============================================================================

def compute_binary_policy_metrics(y_true: pd.Series, scores: np.ndarray, threshold: float) -> Dict[str, Any]:
    """Compute threshold-level binary metrics."""
    y_pred = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[NEGATIVE_LABEL, POSITIVE_LABEL]).ravel()
    return {
        "threshold": float(threshold),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "predicted_positive_count": int(y_pred.sum()),
        "predicted_positive_rate": float(y_pred.mean()),
    }


def build_threshold_sweep_table(y_true: pd.Series, scores: np.ndarray, thresholds: np.ndarray) -> pd.DataFrame:
    """Build threshold sweep table."""
    return pd.DataFrame([
        compute_binary_policy_metrics(y_true, scores, float(threshold))
        for threshold in thresholds
    ])


def select_threshold_by_policy(
    threshold_table: pd.DataFrame,
    policy: str = "max_f1",
    target_recall_floor: float = 0.80,
    max_alert_rate: float = 0.02,
) -> pd.Series:
    """Select threshold by explicit policy."""
    require_columns(threshold_table, ["threshold", "precision", "recall", "f1", "predicted_positive_rate"], "threshold_table")
    if policy == "max_f1":
        return threshold_table.sort_values(["f1", "recall", "precision"], ascending=[False, False, False]).iloc[0]
    if policy == "max_precision_with_recall_floor":
        candidates = threshold_table[threshold_table["recall"] >= target_recall_floor]
        if candidates.empty:
            raise ValueError("No threshold satisfies recall floor.")
        return candidates.sort_values(["precision", "f1", "threshold"], ascending=[False, False, False]).iloc[0]
    if policy == "max_recall_with_alert_cap":
        candidates = threshold_table[threshold_table["predicted_positive_rate"] <= max_alert_rate]
        if candidates.empty:
            raise ValueError("No threshold satisfies alert-rate cap.")
        return candidates.sort_values(["recall", "precision", "f1"], ascending=[False, False, False]).iloc[0]
    raise ValueError(f"Unsupported policy: {policy}")


threshold_grid = np.unique(np.concatenate([
    np.linspace(0.001, 0.050, 50),
    np.linspace(0.055, 0.500, 90),
    np.linspace(0.510, 0.990, 49),
]))

threshold_sweep_df = build_threshold_sweep_table(y_train, oof_scores, threshold_grid)

display(threshold_sweep_df.sort_values("f1", ascending=False).head(15))

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.plot(threshold_sweep_df["threshold"], threshold_sweep_df["precision"], linewidth=2, label="Precision")
ax.plot(threshold_sweep_df["threshold"], threshold_sweep_df["recall"], linewidth=2, label="Recall")
ax.plot(threshold_sweep_df["threshold"], threshold_sweep_df["f1"], linewidth=2, label="F1")
ax.axvline(0.50, linestyle="--", linewidth=1.5, label="Threshold 0.50")
ax.set_title("Trade-off precision / recall / F1 según threshold")
ax.set_xlabel("Threshold")
ax.set_ylabel("Valor")
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.plot(threshold_sweep_df["threshold"], threshold_sweep_df["predicted_positive_count"], linewidth=2)
ax.axvline(0.50, linestyle="--", linewidth=1.5, label="Threshold 0.50")
ax.set_title("Volumen de alertas según threshold")
ax.set_xlabel("Threshold")
ax.set_ylabel("Número de casos alertados")
ax.legend()
plt.show()

THRESHOLD_POLICY = "max_f1"
TARGET_RECALL_FLOOR = 0.80
MAX_ALERT_RATE = 0.02

selected_threshold_row = select_threshold_by_policy(
    threshold_table=threshold_sweep_df,
    policy=THRESHOLD_POLICY,
    target_recall_floor=TARGET_RECALL_FLOOR,
    max_alert_rate=MAX_ALERT_RATE,
)

SELECTED_THRESHOLD = float(selected_threshold_row["threshold"])
selected_threshold_metrics = compute_binary_policy_metrics(y_train, oof_scores, SELECTED_THRESHOLD)
default_threshold_metrics = compute_binary_policy_metrics(y_train, oof_scores, 0.50)

threshold_policy_comparison = pd.DataFrame([
    {"policy": "default_0.50", **default_threshold_metrics},
    {"policy": THRESHOLD_POLICY, **selected_threshold_metrics},
])

display(threshold_policy_comparison)

pretest_decision_record = pd.DataFrame([{
    "candidate_model": CANDIDATE_MODEL_NAME,
    "cv_pr_auc_mean": CANDIDATE_PR_AUC_MEAN,
    "cv_pr_auc_std": CANDIDATE_PR_AUC_STD,
    "oof_pr_auc": OOF_PR_AUC,
    "oof_roc_auc": OOF_ROC_AUC,
    "threshold_policy": THRESHOLD_POLICY,
    "selected_threshold": SELECTED_THRESHOLD,
    "oof_precision": selected_threshold_metrics["precision"],
    "oof_recall": selected_threshold_metrics["recall"],
    "oof_f1": selected_threshold_metrics["f1"],
    "test_set_used": False,
}])

display(pretest_decision_record)

results_registry["threshold_sweep_df"] = threshold_sweep_df
results_registry["threshold_policy_comparison"] = threshold_policy_comparison
results_registry["pretest_decision_record"] = pretest_decision_record


## 14. Evaluación final en test reservado

El test set se usa una sola vez, después de congelar:

- candidato;
- métrica principal;
- estrategia de desbalance;
- HPO;
- threshold.

No se recalcula threshold en test.


In [ ]:
# =============================================================================
# 14.1 Final fit and test scores
# =============================================================================

print_section("FINAL MODEL FIT")

final_model = clone(candidate_model)
final_fit_start_time = time.time()
final_model.fit(X_train, y_train)
final_fit_time_sec = time.time() - final_fit_start_time

print(f"Final candidate model: {CANDIDATE_MODEL_NAME}")
print(f"Final fit time: {final_fit_time_sec:.2f} seconds")


def get_positive_class_scores_from_fitted_model(fitted_estimator: Any, X_data: pd.DataFrame) -> np.ndarray:
    """Extract positive-class scores from a fitted classifier."""
    if hasattr(fitted_estimator, "predict_proba"):
        proba = fitted_estimator.predict_proba(X_data)
        if proba.ndim != 2 or proba.shape[1] < 2:
            raise ValueError("predict_proba did not return a two-column matrix.")
        scores = proba[:, 1]
    elif hasattr(fitted_estimator, "decision_function"):
        raw_scores = np.asarray(fitted_estimator.decision_function(X_data), dtype=float)
        scores = (raw_scores - raw_scores.min()) / (raw_scores.max() - raw_scores.min())
    else:
        raise AttributeError("Estimator has neither predict_proba nor decision_function.")
    scores = np.asarray(scores, dtype=float)
    if np.isnan(scores).any():
        raise ValueError("Extracted scores contain NaN values.")
    return scores


y_test_scores = get_positive_class_scores_from_fitted_model(final_model, X_test)

TEST_PR_AUC = float(average_precision_score(y_test, y_test_scores))
TEST_ROC_AUC = float(roc_auc_score(y_test, y_test_scores))

test_ranking_summary = pd.DataFrame([{
    "candidate_model": CANDIDATE_MODEL_NAME,
    "test_pr_auc": TEST_PR_AUC,
    "test_roc_auc": TEST_ROC_AUC,
    "selected_threshold": SELECTED_THRESHOLD,
    "threshold_policy": THRESHOLD_POLICY,
    "test_set_used_for_selection": False,
}])

display(test_ranking_summary)


In [ ]:
# =============================================================================
# 14.2 Test curves, threshold metrics and confusion matrix
# =============================================================================

test_precision_curve, test_recall_curve, test_pr_thresholds = precision_recall_curve(y_test, y_test_scores)
test_fpr_curve, test_tpr_curve, test_roc_thresholds = roc_curve(y_test, y_test_scores)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(test_recall_curve, test_precision_curve, linewidth=2, label=f"Test PR, AP={TEST_PR_AUC:.4f}")
ax.axhline(float(y_test.mean()), linestyle="--", linewidth=1.5, label="Prevalencia test")
ax.set_title("Curva Precision-Recall en test reservado")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(test_fpr_curve, test_tpr_curve, linewidth=2, label=f"Test ROC, AUC={TEST_ROC_AUC:.4f}")
ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1.5, label="Referencia aleatoria")
ax.set_title("Curva ROC en test reservado")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend()
plt.show()

test_default_threshold_metrics = compute_binary_policy_metrics(y_test, y_test_scores, 0.50)
test_selected_threshold_metrics = compute_binary_policy_metrics(y_test, y_test_scores, SELECTED_THRESHOLD)

test_threshold_comparison = pd.DataFrame([
    {"candidate_model": CANDIDATE_MODEL_NAME, "policy": "default_0.50", **test_default_threshold_metrics},
    {"candidate_model": CANDIDATE_MODEL_NAME, "policy": THRESHOLD_POLICY, **test_selected_threshold_metrics},
])

display(test_threshold_comparison)

y_test_pred_selected = (y_test_scores >= SELECTED_THRESHOLD).astype(int)
test_cm_selected = confusion_matrix(y_test, y_test_pred_selected, labels=[NEGATIVE_LABEL, POSITIVE_LABEL])
test_cm_selected_df = pd.DataFrame(
    test_cm_selected,
    index=["actual_no_fraud", "actual_fraud"],
    columns=["pred_no_fraud", "pred_fraud"],
)

display(test_cm_selected_df)

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(confusion_matrix=test_cm_selected, display_labels=["No fraude", "Fraude"]).plot(ax=ax, values_format="d")
ax.set_title(f"Matriz de confusión en test — threshold={SELECTED_THRESHOLD:.4f}")
plt.show()

print_section("CLASSIFICATION REPORT — TEST SET")
print(classification_report(y_test, y_test_pred_selected, target_names=["no_fraud", "fraud"], zero_division=0))


## 15. Tabla final, importancia de variables y conclusión defendible


In [ ]:
# =============================================================================
# 15.1 Final evidence table and feature importance
# =============================================================================

final_evidence_table = pd.DataFrame([
    {
        "stage": "cross_validation_selection",
        "model": CANDIDATE_MODEL_NAME,
        "pr_auc": CANDIDATE_PR_AUC_MEAN,
        "pr_auc_std": CANDIDATE_PR_AUC_STD,
        "roc_auc": np.nan,
        "threshold": np.nan,
        "precision": np.nan,
        "recall": np.nan,
        "f1": np.nan,
        "predicted_positive_count": np.nan,
        "predicted_positive_rate": np.nan,
    },
    {
        "stage": "out_of_fold_threshold_design",
        "model": CANDIDATE_MODEL_NAME,
        "pr_auc": OOF_PR_AUC,
        "pr_auc_std": np.nan,
        "roc_auc": OOF_ROC_AUC,
        "threshold": SELECTED_THRESHOLD,
        "precision": selected_threshold_metrics["precision"],
        "recall": selected_threshold_metrics["recall"],
        "f1": selected_threshold_metrics["f1"],
        "predicted_positive_count": selected_threshold_metrics["predicted_positive_count"],
        "predicted_positive_rate": selected_threshold_metrics["predicted_positive_rate"],
    },
    {
        "stage": "reserved_test_final_evaluation",
        "model": CANDIDATE_MODEL_NAME,
        "pr_auc": TEST_PR_AUC,
        "pr_auc_std": np.nan,
        "roc_auc": TEST_ROC_AUC,
        "threshold": SELECTED_THRESHOLD,
        "precision": test_selected_threshold_metrics["precision"],
        "recall": test_selected_threshold_metrics["recall"],
        "f1": test_selected_threshold_metrics["f1"],
        "predicted_positive_count": test_selected_threshold_metrics["predicted_positive_count"],
        "predicted_positive_rate": test_selected_threshold_metrics["predicted_positive_rate"],
    },
])

display(final_evidence_table)


def get_final_estimator_from_pipeline_or_model(fitted_estimator: Any) -> Any:
    """Extract the final estimator from a pipeline if needed."""
    if hasattr(fitted_estimator, "named_steps"):
        if "model" in fitted_estimator.named_steps:
            return fitted_estimator.named_steps["model"]
        return list(fitted_estimator.named_steps.values())[-1]
    return fitted_estimator


def extract_feature_importance(fitted_estimator: Any, feature_names: List[str]) -> pd.DataFrame:
    """Extract feature importance when supported."""
    final_estimator = get_final_estimator_from_pipeline_or_model(fitted_estimator)
    importance_values = None

    if hasattr(final_estimator, "feature_importances_"):
        importance_values = np.asarray(final_estimator.feature_importances_)
    elif hasattr(final_estimator, "get_feature_importance"):
        try:
            importance_values = np.asarray(final_estimator.get_feature_importance())
        except Exception:
            importance_values = None

    if importance_values is None or len(importance_values) != len(feature_names):
        return pd.DataFrame()

    importance_df = pd.DataFrame({"feature": feature_names, "importance": importance_values})
    total = importance_df["importance"].sum()
    importance_df["importance_normalized"] = importance_df["importance"] / total if total > 0 else np.nan
    return importance_df.sort_values("importance", ascending=False).reset_index(drop=True)


feature_importance_df = extract_feature_importance(final_model, X_train.columns.tolist())

if feature_importance_df.empty:
    print("Feature importance is not available for this final estimator.")
else:
    display(feature_importance_df.head(20))
    top_importance = feature_importance_df.head(20).sort_values("importance")
    fig, ax = plt.subplots(figsize=(9, 7))
    ax.barh(top_importance["feature"], top_importance["importance"])
    ax.set_title("Top 20 variables por importancia del modelo final")
    ax.set_xlabel("Importancia")
    ax.set_ylabel("Variable")
    plt.show()


In [ ]:
# =============================================================================
# 15.2 Final experiment record and conclusion
# =============================================================================

final_experiment_record = pd.DataFrame([{
    "course_session": "Sesión 2 — Boosting moderno y HPO con validación correcta",
    "dataset": "Credit Card Fraud Detection",
    "candidate_model": CANDIDATE_MODEL_NAME,
    "candidate_group": CANDIDATE_MODEL_GROUP,
    "primary_metric": "average_precision / PR-AUC",
    "selection_validation": "Stratified K-Fold on training set",
    "threshold_source": "out-of-fold predictions on training set",
    "threshold_policy": THRESHOLD_POLICY,
    "selected_threshold": SELECTED_THRESHOLD,
    "cv_pr_auc_mean": CANDIDATE_PR_AUC_MEAN,
    "cv_pr_auc_std": CANDIDATE_PR_AUC_STD,
    "oof_pr_auc": OOF_PR_AUC,
    "oof_roc_auc": OOF_ROC_AUC,
    "test_pr_auc": TEST_PR_AUC,
    "test_roc_auc": TEST_ROC_AUC,
    "test_precision": test_selected_threshold_metrics["precision"],
    "test_recall": test_selected_threshold_metrics["recall"],
    "test_f1": test_selected_threshold_metrics["f1"],
    "test_predicted_positive_count": test_selected_threshold_metrics["predicted_positive_count"],
    "test_predicted_positive_rate": test_selected_threshold_metrics["predicted_positive_rate"],
    "test_set_used_for_model_selection": False,
    "final_fit_time_sec": final_fit_time_sec,
}])

display(final_experiment_record)

results_registry["test_ranking_summary"] = test_ranking_summary
results_registry["test_threshold_comparison"] = test_threshold_comparison
results_registry["final_evidence_table"] = final_evidence_table
results_registry["feature_importance_df"] = feature_importance_df
results_registry["final_experiment_record"] = final_experiment_record

final_conclusion = f"""
Conclusión técnica final:

Bajo Stratified K-Fold en el conjunto de entrenamiento y usando
average_precision / PR-AUC como métrica principal, el candidato seleccionado fue:

    {CANDIDATE_MODEL_NAME}

Evidencia durante selección:
    CV PR-AUC promedio = {CANDIDATE_PR_AUC_MEAN:.6f}
    CV PR-AUC std      = {CANDIDATE_PR_AUC_STD:.6f}

Evidencia out-of-fold:
    OOF PR-AUC         = {OOF_PR_AUC:.6f}
    OOF ROC-AUC        = {OOF_ROC_AUC:.6f}

Política de threshold:
    policy             = {THRESHOLD_POLICY}
    selected_threshold = {SELECTED_THRESHOLD:.6f}

Evaluación final en test reservado:
    Test PR-AUC        = {TEST_PR_AUC:.6f}
    Test ROC-AUC       = {TEST_ROC_AUC:.6f}
    Test precision     = {test_selected_threshold_metrics['precision']:.6f}
    Test recall        = {test_selected_threshold_metrics['recall']:.6f}
    Test F1            = {test_selected_threshold_metrics['f1']:.6f}
    Alertas en test    = {test_selected_threshold_metrics['predicted_positive_count']:,}
    Tasa de alertas    = {test_selected_threshold_metrics['predicted_positive_rate']:.6f}

Lectura metodológica:
    Esta conclusión es defendible porque el modelo, los hiperparámetros,
    la estrategia de desbalance y el threshold fueron seleccionados antes
    de evaluar el test set. El test set se usó solo como verificación final.

Limitaciones:
    - El threshold usado es una política ilustrativa, no necesariamente una política
      de negocio final.
    - En fraude real habría que incorporar costos, capacidad de revisión y análisis
      temporal o por entidad.
    - La importancia de variables debe leerse con cautela porque el dataset está
      anonimizado y contiene variables transformadas.
    - Antes de operación o PI final, conviene revisar calibración, estabilidad temporal,
      leakage transaccional y análisis de errores.
"""

print(final_conclusion)


## 16. Checklist metodológico final

Antes de declarar una selección como defendible, revisa:

- [x] El dataset fue auditado antes de modelar.
- [x] La clase positiva rara fue identificada explícitamente.
- [x] `accuracy` no fue usada como métrica principal.
- [x] Se creó un test set estratificado y reservado.
- [x] Los baselines se evaluaron antes de modelos complejos.
- [x] XGBoost, LightGBM y CatBoost se compararon bajo el mismo protocolo cuando estuvieron disponibles.
- [x] HPO optimizó `average_precision`, no una métrica débil para desbalance.
- [x] La validación vivió dentro de la función objetivo de HPO.
- [x] SMOTE, cuando se usó, vivió dentro del pipeline y dentro de cada fold.
- [x] El candidato final se seleccionó por evidencia de CV.
- [x] El threshold se seleccionó con out-of-fold predictions.
- [x] El test set se usó solo al final.
- [x] La conclusión nombra métrica, validación, modelo, threshold y limitaciones.


## 17. Auditoría final del notebook

Esta auditoría valida estructura, objetos críticos, integridad de datos, leakage, métricas y alineación pedagógica.


In [ ]:
# =============================================================================
# 17.1 Required objects audit
# =============================================================================

def audit_required_objects(object_names: List[str], scope: Dict[str, Any]) -> pd.DataFrame:
    """Audit whether required objects exist."""
    return pd.DataFrame([
        {
            "object": name,
            "exists": name in scope,
            "type": type(scope[name]).__name__ if name in scope else None,
        }
        for name in object_names
    ])


required_objects = [
    "RANDOM_STATE", "TARGET_COL", "df", "X", "y", "X_train", "X_test", "y_train", "y_test",
    "cv", "CLASSIFICATION_CV_SCORING", "baseline_results", "boosting_base_results",
    "HPO_FAMILY", "best_hpo_params", "best_hpo_model", "tuned_report", "strategy_results",
    "final_cv_comparison", "candidate_model", "CANDIDATE_MODEL_NAME", "oof_scores",
    "OOF_PR_AUC", "OOF_ROC_AUC", "SELECTED_THRESHOLD", "THRESHOLD_POLICY",
    "final_model", "y_test_scores", "TEST_PR_AUC", "TEST_ROC_AUC",
    "final_evidence_table", "final_experiment_record", "results_registry", "model_registry",
]

required_audit = audit_required_objects(required_objects, globals())
display(required_audit)

missing_required_objects = required_audit.query("exists == False")
if not missing_required_objects.empty:
    raise RuntimeError("Some required objects are missing. Run all previous cells in order.")

print("Required objects audit passed.")


In [ ]:
# =============================================================================
# 17.2 Data integrity, leakage and metric audits
# =============================================================================

data_integrity_audit = pd.DataFrame([
    {
        "check": "X_train and y_train length match",
        "passed": len(X_train) == len(y_train),
        "details": f"{len(X_train)} vs {len(y_train)}",
    },
    {
        "check": "X_test and y_test length match",
        "passed": len(X_test) == len(y_test),
        "details": f"{len(X_test)} vs {len(y_test)}",
    },
    {
        "check": "train/test index overlap is empty",
        "passed": len(set(X_train.index).intersection(set(X_test.index))) == 0,
        "details": f"overlap={len(set(X_train.index).intersection(set(X_test.index)))}",
    },
    {
        "check": "positive class exists in train",
        "passed": int(y_train.sum()) > 0,
        "details": f"positive_count={int(y_train.sum())}",
    },
    {
        "check": "positive class exists in test",
        "passed": int(y_test.sum()) > 0,
        "details": f"positive_count={int(y_test.sum())}",
    },
])

display(data_integrity_audit)
if not data_integrity_audit["passed"].all():
    raise RuntimeError("Data integrity audit failed.")

leakage_audit = pd.DataFrame([
    {
        "check": "test set was not used for model selection",
        "passed": bool(final_experiment_record["test_set_used_for_model_selection"].iloc[0] == False),
        "evidence": "final_experiment_record",
    },
    {
        "check": "threshold selected before test",
        "passed": bool(pretest_decision_record["test_set_used"].iloc[0] == False),
        "evidence": "pretest_decision_record",
    },
    {
        "check": "SMOTE strategies, if used, are pipelines",
        "passed": all(hasattr(model, "steps") for name, model in strategy_models.items() if "smote" in name),
        "evidence": "strategy_models",
    },
])

display(leakage_audit)
if not leakage_audit["passed"].all():
    raise RuntimeError("Leakage audit failed.")

metric_values_to_check = {
    "CANDIDATE_PR_AUC_MEAN": CANDIDATE_PR_AUC_MEAN,
    "CANDIDATE_PR_AUC_STD": CANDIDATE_PR_AUC_STD,
    "OOF_PR_AUC": OOF_PR_AUC,
    "OOF_ROC_AUC": OOF_ROC_AUC,
    "SELECTED_THRESHOLD": SELECTED_THRESHOLD,
    "TEST_PR_AUC": TEST_PR_AUC,
    "TEST_ROC_AUC": TEST_ROC_AUC,
    "test_precision": test_selected_threshold_metrics["precision"],
    "test_recall": test_selected_threshold_metrics["recall"],
    "test_f1": test_selected_threshold_metrics["f1"],
}

metric_consistency_audit = pd.DataFrame([
    {
        "metric": name,
        "value": value,
        "is_finite": bool(np.isfinite(value)),
        "in_expected_range": bool(value >= 0 if name == "CANDIDATE_PR_AUC_STD" else 0 <= value <= 1),
        "passed": bool(np.isfinite(value) and (value >= 0 if name == "CANDIDATE_PR_AUC_STD" else 0 <= value <= 1)),
    }
    for name, value in metric_values_to_check.items()
])

display(metric_consistency_audit)
if not metric_consistency_audit["passed"].all():
    raise RuntimeError("Metric consistency audit failed.")

print("Data, leakage and metric audits passed.")


In [ ]:
# =============================================================================
# 17.3 Pedagogical alignment audit
# =============================================================================

pedagogical_alignment = pd.DataFrame([
    {"requirement": "Credit Card Fraud Detection dataset", "evidence": "dataset resolver + target audit", "status": "satisfied"},
    {"requirement": "rare-event metric logic", "evidence": "class balance + PR-AUC as primary metric", "status": "satisfied"},
    {"requirement": "baselines before boosting", "evidence": "baseline_results", "status": "satisfied"},
    {"requirement": "XGBoost/LightGBM/CatBoost comparison", "evidence": "boosting_base_results when libraries are available", "status": "satisfied"},
    {"requirement": "HPO with correct validation", "evidence": "objective uses Stratified K-Fold and average_precision", "status": "satisfied"},
    {"requirement": "Optuna fallback", "evidence": "RandomizedSearchCV fallback", "status": "satisfied"},
    {"requirement": "HPO diagnostics", "evidence": "normalized_hpo_report and plots", "status": "satisfied"},
    {"requirement": "class weights vs SMOTE", "evidence": "strategy_results + safe SMOTE pipeline", "status": "satisfied"},
    {"requirement": "threshold as policy", "evidence": "OOF threshold sweep", "status": "satisfied"},
    {"requirement": "test only at the end", "evidence": "pretest_decision_record + final_experiment_record", "status": "satisfied"},
    {"requirement": "defensible conclusion", "evidence": "final_evidence_table + final_conclusion", "status": "satisfied"},
])

display(pedagogical_alignment)

if not (pedagogical_alignment["status"] == "satisfied").all():
    raise RuntimeError("Pedagogical alignment audit failed.")

complexity_progression = pd.DataFrame([
    {"order": 1, "stage": "contract", "passed": True},
    {"order": 2, "stage": "environment", "passed": True},
    {"order": 3, "stage": "dataset audit", "passed": True},
    {"order": 4, "stage": "baselines", "passed": "baseline_results" in globals()},
    {"order": 5, "stage": "boosting base", "passed": "boosting_base_results" in globals()},
    {"order": 6, "stage": "hpo", "passed": "best_hpo_params" in globals()},
    {"order": 7, "stage": "hpo diagnostics", "passed": "normalized_hpo_report" in globals()},
    {"order": 8, "stage": "imbalance strategies", "passed": "strategy_results" in globals()},
    {"order": 9, "stage": "threshold policy", "passed": "threshold_sweep_df" in globals()},
    {"order": 10, "stage": "test final", "passed": "TEST_PR_AUC" in globals()},
    {"order": 11, "stage": "methodological close", "passed": "final_experiment_record" in globals()},
])

display(complexity_progression)
if not complexity_progression["passed"].all():
    raise RuntimeError("Complexity progression audit failed.")

NOTEBOOK_FINAL_AUDIT_PASSED = True

print_section("FINAL AUDIT PASSED")
print("The notebook is methodologically complete under the Session 2 contract.")


## 18. Ejercicios de extensión

### Ejercicio 1 — Rediseñar la política de threshold

Cambia `THRESHOLD_POLICY` por:

```text
max_precision_with_recall_floor
```

y define otro `TARGET_RECALL_FLOOR`. Responde:

- ¿cuánto sube o baja la precision?
- ¿cuántas alertas se generan?
- ¿qué riesgo operativo cambia?

### Ejercicio 2 — Comparar otra familia en HPO

Si tienes tiempo de ejecución suficiente, fuerza `HPO_FAMILY` a otra familia disponible y repite el bloque de HPO.

Responde:

- ¿la mejora fue real o marginal?
- ¿cambió la variabilidad entre folds?
- ¿la familia justifica su costo computacional?

### Ejercicio 3 — Auditar SMOTE

Cambia `sampling_strategy` en el pipeline de SMOTE:

```text
0.05, 0.10, 0.20
```

y evalúa si más oversampling realmente mejora PR-AUC o solo aumenta complejidad.

### Ejercicio 4 — Llevarlo al PI

Para un problema supervisado tabular de tu PI, escribe:

- métrica principal;
- esquema de validación;
- baseline serio;
- familia boosted candidata;
- estrategia de desbalance;
- política de threshold o criterio de decisión;
- evidencia mínima para declarar un modelo defendible.


## Cierre del notebook

Este notebook materializa la idea central de la sesión:

> Boosting moderno es fuerte cuando el experimento que lo sostiene también es fuerte.

La calidad de la decisión no proviene de usar XGBoost, LightGBM, CatBoost u Optuna por sí solos.

Proviene de construir una comparación donde:

- la métrica corresponde al problema;
- la validación protege la conclusión;
- el tuning no usa test;
- el desbalance se trata como hipótesis, no como receta;
- el threshold se decide con evidencia;
- el resultado final se presenta con límites claros.

Ese es el estándar mínimo para llevar un modelo boosted al taller modular o al PI.


In [ ]:
# =============================================================================
# End of notebook
# =============================================================================

print_section("FIN DEL NOTEBOOK")
print("Boosting moderno y optimización de hiperparámetros con validación correcta")
print("Revisa final_experiment_record para la síntesis final del experimento.")
